# 束搜索（Beam Search）

> 承接上一节：`predict_seq2seq` 每一步用 `argmax` 选概率最高的 Token，这叫**贪心搜索**。束搜索是对它的改进。

## 一、Why：为什么需要束搜索

### 1. 预测的真正目标

Seq2Seq 建模的是条件概率，整句的概率是每一步条件概率的乘积：

$$P(Y \mid X) = \prod_{t=1}^{T} P(y_t \mid y_1, \ldots, y_{t-1}, X)$$

我们真正想要的是**整句概率最高**的输出序列，而不是每一步概率最高的 Token。

### 2. 贪心搜索的问题：每一步最优 ≠ 整句最优

贪心搜索每一步都选当前概率最高的 Token。计算量小，但**不保证得到整句概率最高的序列**。

例子（d2l 书中示例）：

| 搜索路径 | 每步概率 | 整句概率 |
|---|---|---|
| 贪心：A → B → C → `<eos>` | 0.5 × 0.4 × 0.4 × 0.6 | **0.048** |
| 另一条：A → C → B → `<eos>` | 0.5 × 0.3 × 0.6 × 0.6 | **0.054** ✅ 更高 |

第 2 步贪心选了概率最高的 B（0.4），没选 C（0.3）。但选 C 之后，后面几步的概率更高，整句反而更好。**贪心一旦选定就无法回头。**

### 3. 穷举搜索也不行：太慢

列出所有可能的序列，选整句概率最高的，一定能找到最优解。但计算量是指数级：

- 词表大小 $|\mathcal{Y}|$，最大长度 $T'$，需要评估 $|\mathcal{Y}|^{T'}$ 个序列
- 例如 $|\mathcal{Y}| = 10000$，$T' = 10$：要评估 $10^{40}$ 个序列，根本算不完

**束搜索就是在贪心（快但不准）和穷举（准但太慢）之间的折中。**

## 二、What：束搜索是什么

束搜索有一个超参数：**束宽 $k$**（beam size）。

核心思想：**每一步不只保留 1 个最好的，而是保留 $k$ 个最好的候选序列**，给「暂时不是最好、但后面可能更好」的路径留机会。

- $k = 1$：就是贪心搜索
- $k$ 越大：越接近穷举，结果越好，但越慢

## 三、How：束搜索怎么做

### 1. 步骤

1. **第 1 步**：Decoder 输入 `<bos>`，选出概率最高的 $k$ 个 Token，作为 $k$ 条候选序列
2. **之后每一步**：每条候选序列都往后扩展一个 Token，得到 $k \times |\mathcal{Y}|$ 个新序列，从中保留整句概率最高的 $k$ 个
3. **重复**，直到达到最大长度 $T'$
4. **最终**：从所有候选序列中选出评分最高的一个作为输出

### 2. 例子（词表 = {A, B, C, D, E}，$k = 2$，最大长度 3）

| 时间步 | 扩展出的候选数 | 保留的 2 个候选 |
|---|---|---|
| 1 | 5 个 | A、C |
| 2 | 2 × 5 = 10 个 | AB、CE |
| 3 | 2 × 5 = 10 个 | ABD、CED |

最终在 A、C、AB、CE、ABD、CED 这 6 个候选中选评分最高的。

注意第 2 步：AB 来自 A，CE 来自 C。如果是贪心，第 1 步只留 A，C 开头的路径直接就没了。

### 3. 最终评分：为什么要除以长度

整句概率是连乘，序列越长，乘的数越多（都 < 1），概率越小。直接比较的话，模型会偏向短句子。

所以用**取对数后的平均**来打分，并惩罚长度：

$$\text{score} = \frac{1}{L^{\alpha}} \sum_{t=1}^{L} \log P(y_t \mid y_1, \ldots, y_{t-1}, X)$$

- 取 $\log$：连乘变成连加，数值更稳定（避免很多小数相乘变成接近 0）
- $L$：候选序列的长度（不含 `<eos>` 之后的部分）
- $L^{\alpha}$：长度惩罚，通常 $\alpha = 0.75$。$\log$ 概率都是负数，序列越长总和越小，除以 $L^{\alpha}$ 来抵消对长句的不公平

### 4. 计算量对比

| 方法 | 计算量 | 结果质量 |
|---|---|---|
| 贪心搜索 | $O(\lvert\mathcal{Y}\rvert \, T')$ | 快，但不一定最优 |
| 束搜索 | $O(k \, \lvert\mathcal{Y}\rvert \, T')$ | 折中，$k$ 可调 |
| 穷举搜索 | $O(\lvert\mathcal{Y}\rvert^{T'})$ | 最优，但不可行 |

## 四、总结

- **贪心搜索**：每步选最好的，快，但「每步最优 ≠ 整句最优」
- **穷举搜索**：一定最优，但计算量指数爆炸
- **束搜索**：每步保留 $k$ 个最好的候选，用束宽 $k$ 在速度和质量之间取平衡；$k = 1$ 时退化为贪心搜索
- **最终评分**用对数概率的长度归一化，避免偏向短句子

> 注意：束搜索只用在**预测阶段**，不影响训练。训练时用 Teacher Forcing，直接喂正确答案，不需要搜索。